# Findings & Conclusions

**Research question:** is greater internet exposure (self-reported weekly `time_spent_hours`) associated with *worse* ability to identify AI-generated images (`score_pct`, proportion of 20 items correct)?

**Data:** full cleaned survey export, n = 102. One missing `time_spent_hours` value was imputed with the sample median in `src/cleaning.py`.

**Approach:**
- Pooled OLS, `score_pct ~ time_spent_hours + C(gender) + age`, HC3 robust standard errors (`src/analysis.py`).
- Pearson and Spearman correlations between time online and score.
- Mixed-effects model with a random intercept by gender as a sensitivity check. With only two groups the fit is singular, so it is not interpreted.
- Age-range × gender subgroup models (`src/subgroup_modeling.py`). Most subgroups are small (n < 10) and are descriptive only.

The numbers below are loaded live from `outputs/analysis/`.

In [ ]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path('..').resolve()
out = ROOT / 'outputs' / 'analysis'
s = json.loads((out / 'summary.json').read_text())
t = s['ols']['time_spent_hours']
g = s['ols']['gender_male']
a = s['ols']['age']

display(Markdown(f"""## Key numeric results (pooled OLS, n = {s['n']})

- R² = {s['ols']['r2']:.3f}, 5-fold CV RMSE = {s['ols']['cv_rmse']:.3f}
- **Time spent online:** coef = {t['coef']:.2e} (95% CI {t['ci_low']:.4f} to {t['ci_high']:.4f}), p = {t['pval']:.2f}
- Gender (Male vs Female): {g['coef']:+.3f}, p = {g['pval']:.3f}
- Age: {a['coef']:+.4f} per year, p = {a['pval']:.3f}
- Pearson r = {s['pearson']['r']:.3f} (p = {s['pearson']['pval']:.2f}); Spearman ρ = {s['spearman']['r']:.3f} (p = {s['spearman']['pval']:.2f})
"""))
display(pd.read_csv(out / 'pooled_ols_coefficients.csv', index_col=0))

In [ ]:
# Data diagnostics: sample size, gender split, numeric summary
display(pd.read_csv(out / 'dataset_overview.csv'))
display(pd.read_csv(out / 'gender_counts.csv'))
display(pd.read_csv(out / 'numeric_summary.csv', index_col=0))

In [ ]:
# Full model output
print((out / 'pooled_ols_summary.txt').read_text())
print((out / 'mixed_effects_summary.txt').read_text())

## Conclusion

With the current data (n = 102) there is no evidence that more time online is associated with worse identification of AI images. The time coefficient is essentially zero with a tight confidence interval, and the model explains very little variance in scores. Men scored slightly higher on average (borderline significant); age shows no clear effect.

## Caveats

- Observational, self-reported exposure; not causal.
- One imputed `time_spent_hours` value (median).
- Subgroup analyses involve many comparisons with no correction, and most subgroups are small.
- The mixed-effects model is singular with only two gender groups.

## Reproduce

From the project root:

```bash
python src/run_all.py
```